In [ ]:
import trimesh
import os
import numpy as np
import glob
import struct

from trimesh.sample import sample_surface
from orion.utils.o3d_utils import visualize_o3d_point_cloud, O3DPointCloud
import open3d as o3d

from pygltflib import GLTF2, Scene
 
"""Passes command line arguments into utility function."""
path = "/home/arisrei/research/RPL/ORION/junk/red_building_grm.glb"
output_dir = "/home/arisrei/research/RPL/ORION/visualization/mesh_to_pcd"
scale = 250
os.makedirs(output_dir, exist_ok=True)

mesh = trimesh.load(path, force='mesh')
print(mesh.faces)
points = sample_surface(mesh, mesh.vertices.shape[0])
#points -= points.mean(axis=0)
# output_path = os.path.join(output_dir, os.path.basename(path))
# np.save(output_path, points)
points = points[0]

In [ ]:
glb = GLTF2().load(path)
curr_scene = glb.scenes[glb.scene]
node_index = curr_scene.nodes[0]  # scene.nodes is the indices, not the objects 
box = glb.nodes[node_index]
box.matrix  # will output vertices for the box object
print(box)

In [ ]:
def convert_glb_to_point_cloud(glb_file):
    # Load the glb file
    gltf = GLTF2().load(glb_file)

    # Initialize arrays to store vertices and colors
    vertices = []
    colors = []

    # Extract vertices and colors from mesh primitives
    for mesh in gltf.meshes:
        print(mesh)
        for primitive in mesh.primitives:
            accessor = gltf.accessors[primitive.attributes.POSITION]
            bufferView = gltf.bufferViews[accessor.bufferView]
            buffer = gltf.buffers[bufferView.buffer]
            data = gltf.get_data_from_buffer_uri(buffer.uri)

            for i in range(accessor.count):
                index = bufferView.byteOffset + accessor.byteOffset + i*12  # the location in the buffer of this vertex
                d = data[index:index+12]  # the vertex data
                v = struct.unpack("<fff", d)   # convert from base64 to three floats
                vertices.append(list(v))
                #print(i, v)
            # print(primitive.material)
            # accessor = gltf.accessors[primitive.attributes.COLOR_0]
            # bufferView = gltf.bufferViews[accessor.bufferView]
            # buffer = gltf.buffers[bufferView.buffer]
            # data = gltf.get_data_from_buffer_uri(buffer.uri)

            # for i in range(accessor.count):
            #     index = bufferView.byteOffset + accessor.byteOffset + i*12  # the location in the buffer of this vertex
            #     d = data[index:index+12]  # the vertex data
            #     v = struct.unpack("<fff", d)   # convert from base64 to three floats
            #     colors.append(list(v))
            #     #print(i, v)


    return np.array(vertices), np.array(colors)

In [ ]:
vertices, colors = convert_glb_to_point_cloud(path)
print(vertices.shape)

In [ ]:
pcd_wrapper = O3DPointCloud()
pcd_grm = o3d.geometry.PointCloud()
pcd_grm.points = o3d.utility.Vector3dVector(points)
pcd_wrapper.pcd = pcd_grm

pcd = visualize_o3d_point_cloud(pcd_wrapper)

In [ ]:
colmap_pcd_path = "/home/arisrei/research/RPL/ORION/third_party/Gen6D/data/custom/red_building/object_point_cloud.ply"
colmap_pcd = o3d.io.read_point_cloud(colmap_pcd_path)

pcd_wrapper = O3DPointCloud()
pcd_wrapper.max_points = 40000
pcd_wrapper.pcd = colmap_pcd
#pcd_wrapper.preprocess()

print(np.array(pcd_wrapper.pcd.points).shape)

fig = visualize_o3d_point_cloud(pcd_wrapper)

In [ ]:
pcd_wrapper = O3DPointCloud()
pcd_wrapper.max_points = 80000
pcd_wrapper.pcd = colmap_pcd + pcd_grm
#pcd_wrapper.preprocess()

print(np.array(pcd_wrapper.pcd.points).shape)

fig = visualize_o3d_point_cloud(pcd_wrapper)